# Predicting drug resistance from single-cell expression: ABE dataset

**Inputs**

Run `Perturb-seq_processing_final.ipynb` first. It writes the first two files below to `output/`.

```
├── Perturb-seq_processing_final.ipynb
├── ML_single_cell.ipynb
├── data/
│   ├── zscores-unambiguous-VEPannotated-processed-control.csv   VEP annotations per guide
│   └── 41588_2024_1948_MOESM4_ESM.xlsx                          Supplementary Table 2 of Coelho et al. (2024)
└── output/
    ├── data_checkpoint.csv       processed expression matrix
    ├── double_guide_cells.csv    double-guide cells and the guide responsible for each phenotype
    ├── mofa/                     MOFA+ models, factor matrices and variance explained
    └── figures/
```

Supplementary Table 2 can be downloaded from the paper's Supplementary Information:

> Coelho, M. A. *et al.* Base editing screens define the genetic landscape of cancer drug resistance mechanisms.
> *Nat Genet* **56**, 2479–2492 (2024).

The VEP annotation table (`zscores-unambiguous-VEPannotated-processed-control.csv`) is included in this repository's `data/` folder.

## Setup

In [1]:
import os
import random
from pathlib import Path

import anndata
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from mofapy2.run.entry_point import mofa
from optuna.samplers import TPESampler
from sklearn.metrics import (accuracy_score, auc, confusion_matrix, f1_score,
                             precision_recall_curve, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder

data_dir = Path("data")
output_dir = Path("output")
mofa_dir = output_dir / "mofa"
figure_dir = output_dir / "figures"
mofa_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

seed = 42
random.seed(seed)
np.random.seed(seed)
os.environ["PYTHONHASHSEED"] = str(seed)

factor_counts = [5, 10, 15, 20, 25]   # numbers of MOFA+ factors to compare
n_iterations = 10                     # repeated train/test splits
n_folds = 5                           # cross-validation folds within each training set
test_size = 0.2

## Load processed data

In [2]:
processed_data = pd.read_csv(output_dir / "data_checkpoint.csv", index_col=0)
double_guide_cells = pd.read_csv(output_dir / "double_guide_cells.csv", index_col=0)

metadata_cols = ["variant_class_highest_impact", "gRNA", "resistance_vs_control"]
gene_expression = processed_data.drop(columns=metadata_cols)

assert gene_expression.select_dtypes(exclude="number").empty, \
    f"Non-numeric columns: {gene_expression.select_dtypes(exclude='number').columns.tolist()}"

label_encoder = LabelEncoder().fit(["control", "resistance"])
y = label_encoder.transform(processed_data["resistance_vs_control"])

print(f"Cells: {gene_expression.shape[0]}, genes: {gene_expression.shape[1]}")
print("Classes:", dict(zip(label_encoder.classes_, np.bincount(y))))

Cells: 7529, genes: 13077
Classes: {np.str_('control'): np.int64(6656), np.str_('resistance'): np.int64(873)}


## MOFA+ factors

MOFA+ summarises thousands of genes with a small number of latent factors, reducing the risk of
overfitting. Models are trained for each factor count, and the variance explained (R²) by each
factor is recorded.

In [ ]:
adata = anndata.AnnData(gene_expression.values.astype(np.float64))
adata.obs_names = gene_expression.index
adata.var_names = gene_expression.columns

# Total sum of squares for R², computed once as it depends only on the expression matrix
x_raw = adata.X
ss_tot = np.sum((x_raw - x_raw.mean(axis=0)) ** 2)

def compute_r2(factors, x_raw, ss_tot):
    """Percentage of total variance explained by each factor on its own."""
    r2 = []
    for i in range(factors.shape[1]):
        factor = factors[:, [i]]
        weights = np.linalg.lstsq(factor, x_raw, rcond=None)[0]
        ss_res = np.sum((x_raw - factor @ weights) ** 2)
        r2.append((1 - ss_res / ss_tot) * 100)
    return pd.Series(r2, index=[f"Factor{i + 1}" for i in range(factors.shape[1])])

z_matrices = {}
r2_results = {}

for n in factor_counts:
    print(f"\nTraining MOFA+ with {n} factors…")
    mofa(
        adata,
        expectations=["W", "Z", "AlphaW", "AlphaZ"],
        use_raw=False,
        n_factors=n,
        seed=seed,
        outfile=str(mofa_dir / f"mofa_{n}_factors.hdf5"),
        quiet=False,
    )

    # adata.obsm["X_mofa"] is overwritten by each run, so copy it out
    z_matrices[n] = pd.DataFrame(
        adata.obsm["X_mofa"],
        index=adata.obs_names,
        columns=[f"Factor{i + 1}" for i in range(n)],
    )
    r2_results[n] = compute_r2(adata.obsm["X_mofa"], x_raw, ss_tot)
    print(f"  Total R²: {r2_results[n].sum():.2f}%")

    z_matrices[n].to_csv(mofa_dir / f"z_matrix_{n}_factors.csv")
    r2_results[n].to_csv(mofa_dir / f"r2_{n}_factors.csv")


Training MOFA+ with 5 factors…

        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


Loaded view='rna' group='group1' with N=7529 samples and D=13077 features...


Model options:
- Automatic Relevance Determination prior on the factors: True
- Automatic Relevance Determination prior on the weights: True
- Spike-and-slab prior on the factors: False
- Spike-and-slab prior on the weights: True
Likelihoods:
- View 0 (rna): gaussian




#####

## VEP features for each cell's causal guide

Each cell is linked to the guide responsible for its phenotype: its own guide for single-guide
cells, and for double-guide cells the guide whose class matches the cell's highest-impact class
(from the processing notebook). Guide IDs are mapped to sequences using Supplementary Table 2,
and the guide's VEP annotations are attached.

In [ ]:
vep_table = pd.read_csv(data_dir / "zscores-unambiguous-VEPannotated-processed-control.csv.gz")
vep_table = vep_table[(vep_table["Editor"] == "ABE") & (vep_table["Drug"] == "DEB-CET")]
vep_table = vep_table.drop(columns=["Source"])
vep_table = vep_table.drop(columns=vep_table.columns[1:5])

vep_cols_to_drop = [
    "zscore", "Hit_class", "Target Transcript ID",
    "RefSeq match transcript (MANE Select)",
    "Amino Acid Edits", "Mutation_Category_enc",
]
vep_features = (vep_table.set_index("Guide")
                .drop(columns=[c for c in vep_cols_to_drop if c in vep_table.columns]))

assert vep_features.index.is_unique, "Some guides have more than one row in the VEP table"

In [ ]:
supp_table2 = pd.read_excel(data_dir / "41588_2024_1948_MOESM4_ESM.xlsx", sheet_name=2)
supp_table2_abe = supp_table2[supp_table2["editor"] == "ABE"]
id_to_sequence = supp_table2_abe.set_index("scRNAseq_ID")["guide"].to_dict()

# Causal guide: own guide for single-guide cells, responsible guide for double-guide cells
causal_guide = processed_data["gRNA"].copy()
causal_guide.loc[double_guide_cells.index] = double_guide_cells["responsible_for_phenotype"]
n_unresolved = double_guide_cells["responsible_for_phenotype"].isna().sum()
print(f"Double-guide cells with no identifiable responsible guide: {n_unresolved}")

causal_sequence = causal_guide.map(id_to_sequence)
cell_vep = vep_features.reindex(causal_sequence.values)
cell_vep.index = processed_data.index

n_missing = cell_vep.isna().all(axis=1).sum()
print(f"Cells with no VEP match: {n_missing}/{len(cell_vep)} ({n_missing / len(cell_vep):.1%})")

z_matrices_vep = {n: pd.concat([z_matrices[n], cell_vep], axis=1) for n in factor_counts}

## Train/test splits

The same splits are used for hyperparameter tuning and for every model, so test cells are never
used for tuning and all feature sets are compared on identical cells. Each split keeps the original
class balance in the test set, then downsamples controls in the training set to match the number
of resistance cells.

In [ ]:
def make_split(y, iteration):
    """Return (balanced training indices, test indices) for one iteration."""
    rng = np.random.RandomState(seed + iteration)
    train_idx, test_idx = train_test_split(
        np.arange(len(y)), test_size=test_size, stratify=y, random_state=seed + iteration
    )
    control_idx = train_idx[y[train_idx] == 0]
    resistance_idx = train_idx[y[train_idx] == 1]
    sampled_control_idx = rng.choice(control_idx, size=len(resistance_idx), replace=False)
    balanced_train_idx = np.concatenate([sampled_control_idx, resistance_idx])
    rng.shuffle(balanced_train_idx)
    return balanced_train_idx, test_idx

splits = [make_split(y, i) for i in range(n_iterations)]
train_idx, test_idx = splits[0]
print(f"Training set: {np.bincount(y[train_idx])} (control, resistance)")
print(f"Test set:     {np.bincount(y[test_idx])} (control, resistance)")

## Hyperparameter search (Optuna)

In [ ]:
run_optuna = True

saved_best_params = {
    "n_estimators": 2500,
    "max_depth": 7,
    "learning_rate": 0.04049591223793557,
    "subsample": 0.5081104452195184,
    "colsample_bytree": 0.846498951688699,
    "reg_lambda": 4.979780015669281,
    "reg_alpha": 3.4904832000270054,
    "min_child_weight": 1,
    "gamma": 0.5680586187369752,
}

fixed_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "early_stopping_rounds": 50,
    "random_state": seed,
}

def objective(trial):
    params = {
        **fixed_params,
        "n_estimators": trial.suggest_int("n_estimators", 500, 3000, step=500),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }
    x = z_matrices[15].values.astype(np.float32)
    iteration_aucs = []
    for iteration, (train_idx, _) in enumerate(splits):
        x_train, y_train = x[train_idx], y[train_idx]
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed + iteration)
        fold_aucs = []
        for tr, val in skf.split(x_train, y_train):
            model = xgb.XGBClassifier(**params)
            model.fit(x_train[tr], y_train[tr], eval_set=[(x_train[val], y_train[val])], verbose=False)
            fold_aucs.append(roc_auc_score(y_train[val], model.predict_proba(x_train[val])[:, 1]))
        iteration_aucs.append(np.mean(fold_aucs))
    return np.mean(iteration_aucs)

if run_optuna:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=seed))
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    best_params = study.best_params
    print(f"Best CV AUC: {study.best_value:.4f}")
else:
    best_params = saved_best_params

params = {**fixed_params, **best_params}
print("Parameters:", best_params)

## XGBoost pipeline

For each split: cross-validation on the balanced training set (with early stopping) sets the
number of trees, a final model is trained on the whole training set, and it is evaluated on the
untouched test set.

In [ ]:
def run_xgb_pipeline(features, params, label=""):
    x = features.values.astype(np.float32)
    params_no_es = {k: v for k, v in params.items() if k != "early_stopping_rounds"}
    all_results = []

    for iteration, (train_idx, test_idx) in enumerate(splits):
        x_train, y_train = x[train_idx], y[train_idx]
        x_test, y_test = x[test_idx], y[test_idx]

        # Cross-validation with early stopping to choose the number of trees
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed + iteration)
        fold_train_aucs, fold_val_aucs, fold_best_iters = [], [], []
        for tr, val in skf.split(x_train, y_train):
            model = xgb.XGBClassifier(**params)
            model.fit(x_train[tr], y_train[tr],
                      eval_set=[(x_train[tr], y_train[tr]), (x_train[val], y_train[val])],
                      verbose=False)
            fold_train_aucs.append(roc_auc_score(y_train[tr], model.predict_proba(x_train[tr])[:, 1]))
            fold_val_aucs.append(roc_auc_score(y_train[val], model.predict_proba(x_train[val])[:, 1]))
            fold_best_iters.append(model.best_iteration)

        # Small buffer, as the full training set is larger than each CV training fold
        final_n_estimators = int(np.mean(fold_best_iters)) + 50
        print(f"  [{label}] Iteration {iteration + 1}/{n_iterations}: "
              f"train AUC {np.mean(fold_train_aucs):.4f} | val AUC {np.mean(fold_val_aucs):.4f} | "
              f"trees {final_n_estimators}")

        # Final model on the full balanced training set
        final_model = xgb.XGBClassifier(**{**params_no_es, "n_estimators": final_n_estimators})
        final_model.fit(x_train, y_train, verbose=False)

        y_pred = final_model.predict(x_test)
        y_pred_proba = final_model.predict_proba(x_test)

        collapsed = len(np.unique(y_pred)) == 1
        if collapsed:
            print(f"  WARNING: model predicted only '{label_encoder.classes_[y_pred[0]]}' "
                  f"in iteration {iteration + 1} [{label}]")

        f1_per_class = f1_score(y_test, y_pred, average=None, labels=[0, 1], zero_division=0)
        pr_auc = {}
        for cls_name, cls_idx in [("control", 0), ("resistance", 1)]:
            prec, rec, _ = precision_recall_curve((y_test == cls_idx).astype(int),
                                                  y_pred_proba[:, cls_idx])
            pr_auc[cls_name] = auc(rec, prec)

        all_results.append({
            "iteration": iteration + 1,
            "collapsed": collapsed,
            "fold_train_aucs": fold_train_aucs,
            "fold_val_aucs": fold_val_aucs,
            "fold_best_iters": fold_best_iters,
            "final_n_estimators": final_n_estimators,
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
            "test_auc": roc_auc_score(y_test, y_pred_proba[:, 1]),
            "confusion_matrix": confusion_matrix(y_test, y_pred),
            "f1_control": f1_per_class[0],
            "f1_resistance": f1_per_class[1],
            "pr_auc_control": pr_auc["control"],
            "pr_auc_resistance": pr_auc["resistance"],
            "y_true": y_test,
            "y_pred_proba": y_pred_proba,
        })

    n_collapsed = sum(r["collapsed"] for r in all_results)
    if n_collapsed:
        print(f"\n  [{label}] WARNING: {n_collapsed}/{n_iterations} iterations predicted a single class.")

    def stat(key, fn):
        return fn([r[key] for r in all_results])

    return {
        "all_results": all_results,
        "n_collapsed": n_collapsed,
        "mean_f1_control": stat("f1_control", np.mean),
        "std_f1_control": stat("f1_control", np.std),
        "mean_f1_resistance": stat("f1_resistance", np.mean),
        "std_f1_resistance": stat("f1_resistance", np.std),
        "mean_pr_auc_control": stat("pr_auc_control", np.mean),
        "std_pr_auc_control": stat("pr_auc_control", np.std),
        "mean_pr_auc_resistance": stat("pr_auc_resistance", np.mean),
        "std_pr_auc_resistance": stat("pr_auc_resistance", np.std),
        "macro_pr_auc": np.mean([(r["pr_auc_control"] + r["pr_auc_resistance"]) / 2
                                 for r in all_results]),
    }

## Run the pipeline across all factor counts

In [ ]:
xgb_results = {}
xgb_results_vep = {}

for n in factor_counts:
    print(f"\n{'=' * 60}\nMOFA+ factors only: {n} factors\n{'=' * 60}")
    xgb_results[n] = run_xgb_pipeline(z_matrices[n], params, label=f"{n} factors")
    print(f"\n{'=' * 60}\nMOFA+ factors + VEP: {n} factors\n{'=' * 60}")
    xgb_results_vep[n] = run_xgb_pipeline(z_matrices_vep[n], params, label=f"{n} factors + VEP")

## Summary table

In [ ]:
def mean_std(values):
    return f"{np.mean(values):.4f} ± {np.std(values):.4f}"

summary_rows = []
for n in factor_counts:
    for results, tag in [(xgb_results, "Base"), (xgb_results_vep, "VEP")]:
        res = results[n]
        runs = res["all_results"]
        summary_rows.append({
            "N factors": n,
            "Features": tag,
            "Test AUC": mean_std([r["test_auc"] for r in runs]),
            "Val AUC": mean_std([np.mean(r["fold_val_aucs"]) for r in runs]),
            "Macro F1": mean_std([r["test_macro_f1"] for r in runs]),
            "Accuracy": mean_std([r["test_accuracy"] for r in runs]),
            "PR AUC control": f"{res['mean_pr_auc_control']:.4f} ± {res['std_pr_auc_control']:.4f}",
            "PR AUC resistance": f"{res['mean_pr_auc_resistance']:.4f} ± {res['std_pr_auc_resistance']:.4f}",
            "Macro PR AUC": f"{res['macro_pr_auc']:.4f}",
        })

summary_df = pd.DataFrame(summary_rows).set_index(["N factors", "Features"])
summary_df.to_csv(output_dir / "xgb_summary.csv")
print(summary_df.to_string())

## Macro F1 and overfitting gap: MOFA+ factors vs MOFA+ factors + VEP features

(a) Mean macro F1 on the test sets. (b) Gap between training and validation AUC in cross-validation,
as an overfitting diagnostic.

In [ ]:
def get_means_stds(results, metric_key="test_macro_f1"):
    values = [[r[metric_key] for r in results[n]["all_results"]] for n in factor_counts]
    return np.array([np.mean(v) for v in values]), np.array([np.std(v) for v in values])

def get_gap_stats(results):
    gaps = [[np.mean(r["fold_train_aucs"]) - np.mean(r["fold_val_aucs"])
             for r in results[n]["all_results"]] for n in factor_counts]
    return np.array([np.mean(g) for g in gaps]), np.array([np.std(g) for g in gaps])

means_base, stds_base = get_means_stds(xgb_results)
means_vep, stds_vep = get_means_stds(xgb_results_vep)
gap_base, std_gap_base = get_gap_stats(xgb_results)
gap_vep, std_gap_vep = get_gap_stats(xgb_results_vep)

font_size = 14
bar_width = 0.35
x = np.arange(len(factor_counts))
colours = [("#55A868", "#B07D4C"), ("#4C72B0", "#E8735A")]

plt.rcParams.update({"font.size": font_size})
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, (ax, (means_a, stds_a), (means_b, stds_b), ylabel, ylim, (colour_a, colour_b)) in enumerate(zip(
    axes,
    [(means_base, stds_base), (gap_base, std_gap_base)],
    [(means_vep, stds_vep), (gap_vep, std_gap_vep)],
    ["Mean macro F1", "AUC gap (train − validation)"],
    [(0, 1.05), (0, 0.17)],
    colours,
)):
    bars_a = ax.bar(x - bar_width / 2, means_a, bar_width, yerr=stds_a, label="MOFA+ factors",
                    color=colour_a, alpha=0.85, capsize=5, error_kw={"lw": 1.5})
    bars_b = ax.bar(x + bar_width / 2, means_b, bar_width, yerr=stds_b,
                    label="MOFA+ factors + VEP features",
                    color=colour_b, alpha=0.85, capsize=5, error_kw={"lw": 1.5})

    # Shift value labels sideways so they don't overlap
    offset_a = -bar_width * 0.30 if ax_idx == 0 else 0
    offset_b = 0 if ax_idx == 0 else bar_width * 0.30
    for bars, means, stds, offset in [(bars_a, means_a, stds_a, offset_a),
                                      (bars_b, means_b, stds_b, offset_b)]:
        for bar, m, s in zip(bars, means, stds):
            ax.text(bar.get_x() + bar.get_width() / 2 + offset, m + s + ylim[1] * 0.018,
                    f"{m:.3f}", ha="center", va="bottom", fontsize=font_size - 1)

    ax.set_xticks(x)
    ax.set_xticklabels([str(n) for n in factor_counts], fontsize=font_size)
    ax.set_xlabel("Number of MOFA+ factors", fontsize=font_size + 1, labelpad=8)
    ax.set_ylabel(ylabel, fontsize=font_size + 1, labelpad=8)
    ax.set_ylim(ylim)
    ax.tick_params(axis="y", labelsize=font_size)
    ax.legend(fontsize=font_size - 1, framealpha=0.85)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout(w_pad=4)
plt.savefig(figure_dir / "f1_and_gap_comparison.png", dpi=300, bbox_inches="tight")
plt.show()